##### DataProcessor class

In [4]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split

class DataProcessor:
    def __init__(self, file_path, data_id='data_id', state_id='state_id', test_size=0.2, random_state=42):
        """
        Initializes the DataProcessor, loads the data, processes it, and splits it into training and testing datasets.

        Args:
            file_path (str): Path to the CSV file containing the data.
            data_id (str): Column name for feature data.
            state_id (str): Column name for label data.
            test_size (float): Fraction of the data to be used as the test set.
            random_state (int): Seed for the random number generator.
        """
        self.file_path = file_path
        self.data_id = data_id
        self.state_id = state_id
        self.test_size = test_size
        self.random_state = random_state
        self.train_dataset, self.test_dataset = self.load_and_split_data()

    def process_chunk(self, data, variable):
        """
        Processes a chunk of data and converts it into tensors. The processing differs based on the data type (features or labels).

        Args:
            data (pandas.Series): A series object containing the data to be processed.
            variable (bool): If False, process as feature data (X); if True, process as label data (y).

        Returns:
            List[torch.Tensor]: A list of tensors corresponding to the processed data.
        """
        tensors = []
        if not variable:
            data = data.apply(lambda x: x.replace('\n', '').replace('[', '').replace(']', '').replace('.', '').split())
            data = data.apply(lambda x: [float(num) for num in x])
            tensors = [torch.tensor(entry, dtype=torch.float64).view(20, 3) for entry in data]
        else:
            data = data.apply(lambda x: eval(x))
            tensors = [torch.tensor(entry) for entry in data]
        return tensors

    def data_conversion_csv(self, column_name, variable, chunksize=10000):
        """
        A generator function that reads data from a CSV in chunks and processes it.

        Args:
            column_name (str): The name of the column to read data from.
            variable (bool): Determines how the data should be processed (as features or labels).
            chunksize (int): Number of rows per chunk.

        Yields:
            Iterator over processed data tensors.
        """
        for chunk in pd.read_csv(self.file_path, usecols=[column_name], chunksize=chunksize):
            yield from self.process_chunk(chunk[column_name], variable)

    def load_and_split_data(self):
        """
        Loads and processes data from a CSV file into tensors for features and labels, then splits them into training and testing sets.

        Returns:
            tuple: A tuple containing the training and testing datasets.
        """
        X_data = list(self.data_conversion_csv(self.data_id, False))
        y_data = list(self.data_conversion_csv(self.state_id, True))
        X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=self.test_size, random_state=self.random_state)
        X_train_tensor, X_test_tensor = torch.stack(X_train), torch.stack(X_test)
        y_train_tensor, y_test_tensor = torch.stack(y_train), torch.stack(y_test)
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        return train_dataset, test_dataset

##### DiffLogicNeuralNetwork class

In [5]:
import torch
from torch import nn
from difflogic import LogicLayer, GroupSum

# include input as a dictionary [in_dim, out_dim, ...]
class DiffLogicNeuralNetwork(nn.Module):
    def __init__(self):
        """
        Initializes the DiffLogicNeuralNetwork. This network uses a LogicLayer followed by a GroupSum operation.
        """
        super(DiffLogicNeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.logic_layers = nn.Sequential(
            LogicLayer(
                in_dim=60,              # number of inputs
                out_dim=1770,           # number of outputs
                device='cuda',          # the device (cuda / cpu)
                implementation='cuda',  # the implementation to be used (native cuda / vanilla pytorch)
                connections='random',   # the method for the random initialization of the connections
                grad_factor=2           # for deep models, to avoid vanishing gradients
            ),
        )
        self.group = GroupSum(k=3, tau=30)

    def forward(self, x):
        """
        Defines the forward pass of the network.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after passing through the logic layers and group sum.
        """
        x = self.flatten(x)
        logits = self.logic_layers(x)
        group = self.group(logits)
        return group
    
    def get_logic_gates(self, layer_index=0, node_index=0):
        """
        Retrieves and prints the logic gate for specified neuron in a specified layer.

        Args:
            layer_index (int): Index of the layer from which to extract the logic gate.

        Returns:
            str: Description of the logic gate used by the specified neuron.
        """
        n = self.logic_layers[layer_index].weights.size()[0]
        node_index = node_index
        weights = self.logic_layers[layer_index].weights[node_index]
        gate_index = torch.argmax(weights).item()

        logic_operations = [
            "0", "A and B", "not(A implies B)", "A", "not(B implies A)", "B",
            "A xor B", "A or B", "not(A or B)", "not(A xor B)", "not(B)", "B implies A",
            "not(A)", "A implies B", "not(A and B)", "1"
        ]
        selected_gate = logic_operations[gate_index]
        return f"The logic gate used by neuron {node_index} in layer {layer_index} is: {selected_gate}"

    def save_to_verilog(self, filename="logic_network.v"):
        """
        Generates a Verilog file describing the network's logic gates and their connections.

        Args:
            filename (str): The filename for the output Verilog file.
        """
        logic_gate_verilog = {
            "0": "1'b0",
            "A∧B": "{a} & {b}",
            "¬(A⇒B)": "{a} & ~{b}",
            "A": "{a}",
            "¬(B⇒A)": "{b} & ~{a}",
            "B": "{b}",
            "A⊕B": "{a} ^ {b}",
            "A∨B": "{a} | {b}",
            "¬(A∨B)": "~({a} | {b})",
            "¬(A⊕B)": "~({a} ^ {b})",
            "¬B": "~{b}",
            "B⇒A": "~{b} | {a}",
            "¬A": "~{a}",
            "A⇒B": "~{a} | {b}",
            "¬(A∧B)": "~({a} & {b})",
            "1": "1'b1"
        }
        with open(filename, 'w') as file:
            file.write("module logic_network(\n")
            file.write("    input wire [{}:0] inputs,\n".format(max(max(self.connections.keys()), max(max(con) for con in self.connections.values()))))
            file.write("    output wire [{}:0] outputs\n".format(len(self.neuron_gates) - 1))
            file.write(");\n\n")
            for neuron_id in self.connections:
                a_idx, b_idx = self.connections[neuron_id]
                gate = logic_gate_verilog[self.logic_operations[self.neuron_gates[neuron_id]]].format(a=f"inputs[{a_idx}]", b=f"inputs[{b_idx}]")
                file.write(f"    assign outputs[{neuron_id}] = {gate};\n")
            file.write("endmodule\n")
    
    def save_model(self, path):
        """
        Saves the model to a specified path.

        Args:
            path (str): The path where the model will be saved.
        """
        torch.save(self.state_dict(), path)
        print(f"Model saved to {path}")

    def load_model(self, path):
        """
        Loads the model from a specified path.

        Args:
            path (str): The path from where to load the model.
        """
        self.load_state_dict(torch.load(path))
        self.eval()  # Set the model to evaluation mode
        print(f"Model loaded from {path}")

    def create_logic_network(self):
        """
        Converts the DiffLogic network into a logic network by extracting the most probable logic gate for each neuron.

        Returns:
            LogicNetwork: An instance of the LogicNetwork class representing the operational logic of the model.
        """

        neuron_gates = []
        connections = {}

        for layer in self.logic_layers:
            input_indices = layer.indices[0].cpu().numpy()  # First input indices
            output_indices = layer.indices[1].cpu().numpy()  # Second input indices

            for output_neuron in range(layer.weights.size()[0]):
                gate_idx = torch.argmax(layer.weights[output_neuron]).item()
                neuron_gates.append(gate_idx)
                connections[output_neuron] = (input_indices[output_neuron], output_indices[output_neuron])

        return LogicNetwork(neuron_gates, connections)

##### LogicNetwork class

In [6]:
class LogicNetwork:
    def __init__(self, neuron_gates, connections):
        """
        Initializes a Logic Network from a list of logic gates and their connections.

        Args:
            neuron_gates (list): A list of indices representing logic gates.
            connections (dict): A dictionary mapping each output neuron to its input pair indices.
        """
        self.neurons = {i: LogicNeuron(gate) for i, gate in enumerate(neuron_gates)}
        self.connections = connections

    def forward(self, inputs):
        """
        Computes the forward pass of the Logic Network.

        Args:
            inputs (array): An array of boolean inputs to the network.

        Returns:
            dict: Outputs from each neuron in the network.
        """
        outputs = {}
        for neuron_id, (input_a_idx, input_b_idx) in self.connections.items():
            input_a = inputs[input_a_idx]
            input_b = inputs[input_b_idx]
            outputs[neuron_id] = self.neurons[neuron_id].forward(input_a, input_b)
        return outputs

##### LogicNeuron class

In [7]:
class LogicNeuron:
    def __init__(self, gate_idx):
        """
        Initializes a neuron that performs a specific logic operation based on the gate index.

        Args:
            gate_idx (int): Index of the logic gate operation from a predefined list of functions.
        """
        self.operation = self.get_logic_gate_function(gate_idx)

    def forward(self, input_a, input_b):
        """
        Executes the logic operation of this neuron.

        Args:
            input_a (bool): First input to the logic gate.
            input_b (bool): Second input to the logic gate.

        Returns:
            bool: Output of the logic operation.
        """
        return self.operation(input_a, input_b)

    @staticmethod
    def get_logic_gate_function(index):
        """
        Returns the logic function associated with a specific index.

        Args:
            index (int): Index of the logic function in the predefined list.

        Returns:
            function: Logic function corresponding to the provided index.
        """
        logic_functions = [
            lambda a, b: False,  # Zero
            lambda a, b: a and b,  # AND
            lambda a, b: a and not b,  # NOT IMPLIES
            lambda a, b: a,  # IDENTITY A
            lambda a, b: b and not a,  # NOT IMPLIED BY
            lambda a, b: b,  # IDENTITY B
            lambda a, b: a != b,  # XOR
            lambda a, b: a or b,  # OR
            lambda a, b: not (a or b),  # NOR
            lambda a, b: not (a != b),  # XNOR
            lambda a, b: not b,  # NOT B
            lambda a, b: not a or b,  # IMPLIED BY
            lambda a, b: not a,  # NOT A
            lambda a, b: not b or a,  # IMPLIES
            lambda a, b: not (a and b),  # NAND
            lambda a, b: True  # One
        ]
        return logic_functions[index]

##### Trainer class

In [9]:
import torch
import os
from torch import optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from datetime import datetime

class Trainer:
    def __init__(self, model, loss_fn, optimizer, train_loader, val_loader, device="cuda", patience=5):
        """
        Initializes the Trainer class which handles the training and validation of a model.

        Args:
            model (nn.Module): The neural network model to be trained.
            loss_fn (function): The loss function to be used for training.
            optimizer (torch.optim.Optimizer): The optimizer to use for training.
            train_loader (DataLoader): DataLoader for the training data.
            val_loader (DataLoader): DataLoader for the validation data.
            device (str): Device to use for training ('cuda' or 'cpu').
            patience (int): Patience for early stopping.
        """
        self.model = model.to(device)
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.early_stopping = EarlyStopping(patience=patience, verbose=True)

    def validate(self):
        """
        Performs validation of the model using the validation loader.

        Returns:
            float: The average validation loss.
        """
        self.model.eval()
        total_loss, total = 0, 0
        with torch.no_grad():
            for inputs, targets in self.val_loader:
                inputs, targets = inputs.to(self.device).float(), targets.to(self.device).float()
                outputs = self.model(inputs)
                loss = self.loss_fn(outputs, targets)
                total_loss += loss.item() * inputs.size(0)
                total += inputs.size(0)
        return total_loss / total

    def train(self, num_epochs):
        """
        Trains the model for a specified number of epochs.

        Args:
            num_epochs (int): Number of epochs to train the model.
        """
        for epoch in range(num_epochs):
            self.model.train()
            total_loss, total = 0, 0
            loop = tqdm(self.train_loader, leave=True, desc=f'Epoch {epoch+1}/{num_epochs}')
            for inputs, targets in loop:
                inputs, targets = inputs.to(self.device).float(), targets.to(self.device).float()
                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.loss_fn(outputs, targets)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item() * inputs.size(0)
                total += inputs.size(0)
                loop.set_postfix(loss=(total_loss / total))
            val_loss = self.validate()
            print(f'Epoch {epoch+1}/{num_epochs} - Training Loss: {total_loss / total:.4f}, Validation Loss: {val_loss:.4f}')
            if self.early_stopping(val_loss, self.model):
                print("Early stopping")
                break

##### EarlyStopping class

In [10]:
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0.001):
        """
        Initializes the EarlyStopping mechanism which halts the training process when a certain 
        threshold of non-improvement in validation loss is reached.

        Args:
            patience (int): The number of epochs with no improvement after which training will be stopped.
            verbose (bool): If set to True, it will print out a log message for each validation loss improvement.
            delta (float): The minimum change in the monitored quantity to qualify as an improvement, i.e., 
                           an improvement of less than or equal to 'delta' will count as no improvement.
        """
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')

    def __call__(self, val_loss, model):
        """
        Call method that will be called every epoch to see if the early stopping condition has been met.

        Args:
            val_loss (float): The current validation loss.
            model (nn.Module): The model being trained.

        Returns:
            bool: True if the training should be stopped early, False otherwise.
        """
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
        return False

    def save_checkpoint(self, val_loss, model):
        """
        Saves the model when the validation loss decreases.

        Args:
            val_loss (float): The new lower validation loss.
            model (nn.Module): The model that is being trained.
        """
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        self.val_loss_min = val_loss
        current_date = datetime.now().strftime('%m-%d-%Y')
        filename = f'{current_date}.pth'
        current_path = os.getcwd() + "/saved_model/"
        path = os.path.join(current_path, filename)
        torch.save(model.state_dict(), path)

##### Testing the classes

In [ ]:
import torch
import numpy as np
import os

# Ensure DiffLogicNeuralNetwork, LogicNetwork, and LogicNeuron are already imported.

# dataset_path
dataset_path = "/blue/woodard/share/skeptical_beings/training_datasets/"
big_dataset_path = dataset_path + "schoolhouse_dataset_2024.06.14.csv" # 3GB 
medium_dataset_path = dataset_path + "schoolhouse_dataset_e2_2024.06.14.csv" # 30MB


def main():
    
    # 1. Initialize Data
    data_processor = DataProcessor(medium_dataset_path)
    train_dataset, test_dataset = data_processor.load_and_split_data()

    # 2. Initialize the model
    model = DiffLogicNeuralNetwork().to('cuda')
    print("Model initialized.")

    # 3. Setup DataLoaders
    batch_size = 32  
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 4. Loss function and Optimizer
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 
    
    # 5. Train the model
    trainer = Trainer(model, loss_fn, optimizer, train_loader, val_loader, device="cuda")
    trainer.train(num_epochs=100)  
    
    # 6. Convert to Logic Network with learned gates
    logic_network = model.create_logic_network()

    # 7. Save and loading the model
    model.save_model(path="saved_model/difflogic.pth")
    model.load_model(path="saved_model/difflogic.pth")

    # 8. Other testing other class functions
    number_of_inputs = 60 * 1770
    inputs = np.random.randint(0, 2, size=(number_of_inputs)) 
    outputs = logic_network.forward(inputs)
    print(outputs)
    max_index = max(max(logic_network.connections.values(), key=lambda x: max(x)))
    inputs = np.random.choice([True, False], size=(max_index + 1,))
    print("Random inputs generated for logic network:", inputs)

if __name__ == "__main__":
    main()

#### Model Hyperparameter Testing

In [ ]:
# Define hyperparameters to explore
hyperparams_grid = {
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [16, 32, 64],
    'num_layers': [1, 2]  # Example if you have variable number of layers in your network definition
}

# Training with different hyperparameters
results = []

def run_experiment(hyperparams):
    learning_rate = hyperparams['learning_rate']
    batch_size = hyperparams['batch_size']
    num_layers = hyperparams['num_layers']
    
    # Modify DiffLogic network to accept num_layers as an init parameter
    model = DiffLogicNeuralNetwork(num_layers=num_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Re-create dataloaders with new batch_size
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size)

    # Train and validate the model
    train_losses, val_losses, train_accuracies, val_accuracies = train_loop(
        train_loader, test_loader, model, loss_fn, optimizer, num_epochs=150, patience=5
    )

    final_train_loss = train_losses[-1] if len(train_losses) > 0 else None
    final_val_loss = val_losses[-1] if len(val_losses) > 0 else None
    
    # Store results
    results.append({
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'num_layers': num_layers,
        'final_train_loss': final_train_loss,
        'final_val_loss': final_val_loss
    })

# Run experiments for all combinations of hyperparameters
from itertools import product

for params in product(*hyperparams_grid.values()):
    hyperparams = dict(zip(hyperparams_grid.keys(), params))
    print(f"Running experiment with params: {hyperparams}")
    run_experiment(hyperparams)

# Print all results
for result in results:
    print(result)